In [8]:
import os
from pathlib import Path

import pandas as pd

from classifier import train_xgboost
from feature_combiner import combine_features
from preprocessing import clean_text_for_roberta, clean_text_for_tfidf
from roberta_embedder import extract_embeddings
from tfidf_vectorizer import compute_tfidf, save_vectorizer
from sklearn.preprocessing import StandardScaler

# Set working directory to project root (compatible with Jupyter notebooks)
try:
    project_root = Path(__file__).parent.parent
except NameError:
    # Running in Jupyter notebook
    project_root = Path.cwd()
    if project_root.name == 'src':
        project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

def _validate_schema(df: pd.DataFrame, file_name: str) -> None:
    required = {"content", "label"}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f"{file_name} thiếu cột bắt buộc: {missing}")


def _encode_labels(label_series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(label_series):
        numeric = pd.to_numeric(label_series, errors="coerce")
        if set(numeric.dropna().unique()).issubset({0, 1}):
            return numeric

    normalized = label_series.astype(str).str.strip().str.lower()
    mapping = {
        "truth": 0,
        "rumor": 1,
        "0": 0,
        "1": 1,
    }
    encoded = normalized.map(mapping)

    invalid_mask = encoded.isna()
    if invalid_mask.any():
        invalid_values = sorted(normalized[invalid_mask].unique().tolist())
        raise ValueError(
            "Chi nhan 2 nhan duoc phep: 'truth'->0 va 'rumor'->1. "
            f"Gia tri khong hop le: {invalid_values}"
        )

    return encoded


def _prepare_split(df: pd.DataFrame, split_name: str):
    print(f"\n[{split_name}] cleaning text...")
    df = df.copy()
    df["clean_text_tfidf"] = df["content"].apply(clean_text_for_tfidf)
    df["clean_text_roberta"] = df["content"].apply(clean_text_for_roberta)
    labels = _encode_labels(df["label"])
    valid = (~labels.isna()) & (~df["clean_text_roberta"].isna())

    dropped = int((~valid).sum())
    if dropped > 0:
        print(f"[{split_name}] bỏ {dropped} dòng do label/text không hợp lệ")

    df = df.loc[valid].reset_index(drop=True)
    labels = labels.loc[valid].astype(int).reset_index(drop=True)

    print(f"[{split_name}] samples hợp lệ: {len(df)}")
    print(f"[{split_name}] label distribution:\n{labels.value_counts().sort_index()}")

    return df, labels.values


def main():
    print("Loading train/test data...")
    train_df = pd.read_csv("data/ver_1/train.csv")
    test_df = pd.read_csv("data/ver_1/test.csv")

    _validate_schema(train_df, "train.csv")
    _validate_schema(test_df, "test.csv")

    print(f"Train shape: {train_df.shape}")
    print(f"Test shape:  {test_df.shape}")

    train_df, y_train = _prepare_split(train_df, "TRAIN")
    test_df, y_test = _prepare_split(test_df, "TEST")

    print("\nGenerating TF-IDF features (fit on TRAIN)...")
    X_train_tfidf, tfidf_vectorizer = compute_tfidf(train_df["clean_text_tfidf"])
    X_test_tfidf, _ = compute_tfidf(test_df["clean_text_tfidf"], fitted_vectorizer=tfidf_vectorizer)

    print("Generating RoBERTa embeddings...")
    X_train_roberta = extract_embeddings(train_df["clean_text_roberta"])
    X_test_roberta = extract_embeddings(test_df["clean_text_roberta"])

    # 1. Fit trên train
    scaler_bert = StandardScaler()
    roberta_train_scaled = scaler_bert.fit_transform(X_train_roberta)

    # 2. Transform test
    roberta_test_scaled = scaler_bert.transform(X_test_roberta)

    print("Combining text features...")
    X_train = combine_features(X_train_tfidf, roberta_train_scaled)
    X_test = combine_features(X_test_tfidf, roberta_test_scaled)

    print("Training XGBoost classifier...")
    train_xgboost(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

    print("\nSaving vectorizer...")
    save_vectorizer(tfidf_vectorizer, "tfidf_vectorizer.pkl")
    print("Done.")

if __name__ == "__main__":
    main()


Working directory: d:\HK8\hybrid-fake-news-detector
Loading train/test data...
Train shape: (1711, 4)
Test shape:  (428, 4)

[TRAIN] cleaning text...
[TRAIN] samples hợp lệ: 1711
[TRAIN] label distribution:
label
0    902
1    809
Name: count, dtype: int64

[TEST] cleaning text...
[TEST] samples hợp lệ: 428
[TEST] label distribution:
label
0    256
1    172
Name: count, dtype: int64

Generating TF-IDF features (fit on TRAIN)...
Generating RoBERTa embeddings...


Extracting RoBERTa embeddings: 100%|██████████| 428/428 [00:24<00:00, 17.72it/s]


Combining text features...
Training XGBoost classifier...


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\callback.py:385: UserWarning: [11:40:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


Log loss curves saved to: logloss_train_test.png
Accuracy curves saved to: accuracy_train_test.png
Training curve values saved to: training_curves.json
Best round used for inference: 105

Training complete! Evaluating model...
Feature importance stats:
  Total features: 1803
  Non-zero importances: 589
  Max importance: 0.008265
  Mean importance: 0.000555

Top 20 features by importance:
  1. RoBERTa_dim_642: 0.008265
  2. Feature_1650: 0.006704
  3. Feature_1687: 0.005282
  4. RoBERTa_dim_357: 0.005216
  5. RoBERTa_dim_743: 0.004888
  6. RoBERTa_dim_630: 0.004831
  7. RoBERTa_dim_383: 0.004771
  8. Feature_1594: 0.004623
  9. Feature_1773: 0.004539
  10. RoBERTa_dim_652: 0.004275
  11. RoBERTa_dim_226: 0.004109
  12. RoBERTa_dim_452: 0.004085
  13. RoBERTa_dim_288: 0.003969
  14. RoBERTa_dim_448: 0.003954
  15. RoBERTa_dim_236: 0.003747
  16. RoBERTa_dim_428: 0.003626
  17. Feature_1632: 0.003607
  18. Feature_1749: 0.003558
  19. RoBERTa_dim_645: 0.003486
  20. Feature_1626: 0.003447